# Data Quality Tests - Bronze Layer
## Schema: bike_store.bronze

This notebook performs comprehensive data quality tests on all bronze layer tables:
- **CRM System**: crm_customer, crm_product, crm_sales
- **ERP System**: erp_customer, erp_category, erp_location

### Test Categories:
1. **Completeness**: Null checks on key fields
2. **Schema Consistency**: Column names and types validation
3. **Data Type Validations**: Proper data types for fields
4. **Range Validations**: Dates, amounts, and numeric ranges
5. **Relationship Validations**: Foreign key integrity
6. **Known Issues**: INT dates and monetary values

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime
import pandas as pd

# Initialize test results storage
test_results = []

def log_test(table_name, test_category, test_name, status, details=""):
    """Log test result"""
    test_results.append({
        "table": table_name,
        "category": test_category,
        "test": test_name,
        "status": status,
        "details": details,
        "timestamp": datetime.now()
    })
    
def check_completeness(df, table_name, required_columns):
    """Check for null values in required columns"""
    for col in required_columns:
        if col not in df.columns:
            log_test(table_name, "Completeness", f"{col} exists", "FAIL", f"Column {col} not found")
            continue
            
        null_count = df.filter(F.col(col).isNull()).count()
        total_count = df.count()
        null_pct = (null_count / total_count * 100) if total_count > 0 else 0
        
        if null_count > 0:
            log_test(table_name, "Completeness", f"{col} not null", "WARN", 
                    f"{null_count} nulls ({null_pct:.2f}%)")
        else:
            log_test(table_name, "Completeness", f"{col} not null", "PASS", "No nulls found")

def check_schema(df, table_name, expected_schema):
    """Verify schema matches expectations"""
    actual_columns = set(df.columns)
    expected_columns = set(expected_schema.keys())
    
    # Check missing columns
    missing = expected_columns - actual_columns
    if missing:
        log_test(table_name, "Schema", "All columns present", "FAIL", f"Missing: {missing}")
    else:
        log_test(table_name, "Schema", "All columns present", "PASS", "")
    
    # Check extra columns
    extra = actual_columns - expected_columns
    if extra:
        log_test(table_name, "Schema", "No extra columns", "INFO", f"Extra: {extra}")
    
    # Check data types
    for col, expected_type in expected_schema.items():
        if col in df.columns:
            actual_type = dict(df.dtypes)[col]
            if actual_type.lower() != expected_type.lower():
                log_test(table_name, "Schema", f"{col} type", "FAIL", 
                        f"Expected {expected_type}, got {actual_type}")
            else:
                log_test(table_name, "Schema", f"{col} type", "PASS", "")

def check_range(df, table_name, column, min_val=None, max_val=None, allow_null=True):
    """Check value ranges"""
    if column not in df.columns:
        log_test(table_name, "Range", f"{column} range check", "SKIP", "Column not found")
        return
    
    df_filtered = df if allow_null else df.filter(F.col(column).isNotNull())
    
    if min_val is not None:
        invalid_count = df_filtered.filter(F.col(column) < min_val).count()
        if invalid_count > 0:
            log_test(table_name, "Range", f"{column} >= {min_val}", "FAIL", 
                    f"{invalid_count} values below minimum")
        else:
            log_test(table_name, "Range", f"{column} >= {min_val}", "PASS", "")
    
    if max_val is not None:
        invalid_count = df_filtered.filter(F.col(column) > max_val).count()
        if invalid_count > 0:
            log_test(table_name, "Range", f"{column} <= {max_val}", "FAIL", 
                    f"{invalid_count} values above maximum")
        else:
            log_test(table_name, "Range", f"{column} <= {max_val}", "PASS", "")

print("✅ Helper functions loaded")

## 1. bike_store.bronze.crm_customer
Customer data from CRM system

In [0]:
table_name = "bike_store.bronze.crm_customer"
df = spark.table(table_name)

print(f"Testing {table_name}...")
print(f"Total records: {df.count():,}\n")

# 1. Completeness checks
required_cols = ["cst_id", "cst_key", "cst_firstname", "cst_lastname"]
check_completeness(df, table_name, required_cols)

# 2. Schema consistency
expected_schema = {
    "cst_id": "int",
    "cst_key": "string",
    "cst_firstname": "string",
    "cst_lastname": "string",
    "cst_marital_status": "string",
    "cst_gndr": "string",
    "cst_create_date": "date",
    "file_name": "string",
    "ingest_datetime": "timestamp"
}
check_schema(df, table_name, expected_schema)

# 3. Range validations
check_range(df, table_name, "cst_create_date", max_val=datetime.now().date())

# 4. Data quality checks
# Check for duplicate customer IDs
dup_count = df.groupBy("cst_id").count().filter("count > 1").count()
if dup_count > 0:
    log_test(table_name, "Uniqueness", "cst_id unique", "FAIL", f"{dup_count} duplicate IDs")
else:
    log_test(table_name, "Uniqueness", "cst_id unique", "PASS", "")

# Check gender values
valid_genders = df.filter(F.col("cst_gndr").isin(["M", "F", "Male", "Female"])).count()
total = df.filter(F.col("cst_gndr").isNotNull()).count()
if valid_genders < total:
    log_test(table_name, "Validity", "cst_gndr valid values", "WARN", 
            f"{total - valid_genders} invalid gender values")
else:
    log_test(table_name, "Validity", "cst_gndr valid values", "PASS", "")

print("✅ crm_customer tests completed")

## 2. bike_store.bronze.crm_product
Product data from CRM system

In [0]:
table_name = "bike_store.bronze.crm_product"
df = spark.table(table_name)

print(f"Testing {table_name}...")
print(f"Total records: {df.count():,}\n")

# 1. Completeness checks
required_cols = ["prd_id", "prd_key", "prd_nm"]
check_completeness(df, table_name, required_cols)

# 2. Schema consistency
expected_schema = {
    "prd_id": "int",
    "prd_key": "string",
    "prd_nm": "string",
    "prd_cost": "int",
    "prd_line": "string",
    "prd_start_dt": "date",
    "prd_end_dt": "date",
    "file_name": "string",
    "ingest_datetime": "timestamp"
}
check_schema(df, table_name, expected_schema)

# 3. Range validations
check_range(df, table_name, "prd_cost", min_val=0)
check_range(df, table_name, "prd_start_dt", max_val=datetime.now().date())

# 4. Data quality checks
# Check for duplicate product IDs
dup_count = df.groupBy("prd_id").count().filter("count > 1").count()
if dup_count > 0:
    log_test(table_name, "Uniqueness", "prd_id unique", "FAIL", f"{dup_count} duplicate IDs")
else:
    log_test(table_name, "Uniqueness", "prd_id unique", "PASS", "")

# Check that end date is after start date
invalid_dates = df.filter(
    (F.col("prd_end_dt").isNotNull()) & 
    (F.col("prd_start_dt").isNotNull()) & 
    (F.col("prd_end_dt") < F.col("prd_start_dt"))
).count()
if invalid_dates > 0:
    log_test(table_name, "Logic", "prd_end_dt >= prd_start_dt", "FAIL", 
            f"{invalid_dates} records with end date before start date")
else:
    log_test(table_name, "Logic", "prd_end_dt >= prd_start_dt", "PASS", "")

# Known issue: prd_cost stored as INT (should be DECIMAL)
log_test(table_name, "Known Issues", "prd_cost type", "WARN", 
        "Cost stored as INT - should be DECIMAL in Silver layer")

print("✅ crm_product tests completed")

## 3. bike_store.bronze.crm_sales
Sales transaction data from CRM system

In [0]:
table_name = "bike_store.bronze.crm_sales"
df = spark.table(table_name)

print(f"Testing {table_name}...")
print(f"Total records: {df.count():,}\n")

# 1. Completeness checks
required_cols = ["sls_ord_num", "sls_prd_key", "sls_cust_id", "sls_quantity", "sls_sales"]
check_completeness(df, table_name, required_cols)

# 2. Schema consistency
expected_schema = {
    "sls_ord_num": "string",
    "sls_prd_key": "string",
    "sls_cust_id": "int",
    "sls_order_dt": "int",
    "sls_ship_dt": "int",
    "sls_due_dt": "int",
    "sls_sales": "int",
    "sls_quantity": "int",
    "sls_price": "int",
    "file_name": "string",
    "ingest_datetime": "timestamp"
}
check_schema(df, table_name, expected_schema)

# 3. Range validations
check_range(df, table_name, "sls_sales", min_val=0)
check_range(df, table_name, "sls_quantity", min_val=0)
check_range(df, table_name, "sls_price", min_val=0)

# 4. Relationship validations
# Check FK to crm_customer
customer_df = spark.table("bike_store.bronze.crm_customer")
valid_customer_ids = customer_df.select("cst_id").distinct()
orphaned_sales = df.join(valid_customer_ids, df.sls_cust_id == valid_customer_ids.cst_id, "left_anti").count()
if orphaned_sales > 0:
    log_test(table_name, "Referential Integrity", "sls_cust_id FK valid", "FAIL", 
            f"{orphaned_sales} sales with invalid customer IDs")
else:
    log_test(table_name, "Referential Integrity", "sls_cust_id FK valid", "PASS", "")

# Check FK to crm_product
product_df = spark.table("bike_store.bronze.crm_product")
valid_product_keys = product_df.select("prd_key").distinct()
orphaned_sales_prd = df.join(valid_product_keys, df.sls_prd_key == valid_product_keys.prd_key, "left_anti").count()
if orphaned_sales_prd > 0:
    log_test(table_name, "Referential Integrity", "sls_prd_key FK valid", "FAIL", 
            f"{orphaned_sales_prd} sales with invalid product keys")
else:
    log_test(table_name, "Referential Integrity", "sls_prd_key FK valid", "PASS", "")

# 5. Data quality checks
# Check quantity > 0
zero_qty = df.filter(F.col("sls_quantity") <= 0).count()
if zero_qty > 0:
    log_test(table_name, "Logic", "sls_quantity > 0", "FAIL", f"{zero_qty} records with quantity <= 0")
else:
    log_test(table_name, "Logic", "sls_quantity > 0", "PASS", "")

# Known issues: dates stored as INT
log_test(table_name, "Known Issues", "Date columns type", "WARN", 
        "sls_order_dt, sls_ship_dt, sls_due_dt stored as INT - should be DATE in Silver layer")

# Known issue: monetary values as INT
log_test(table_name, "Known Issues", "Monetary columns type", "WARN", 
        "sls_sales, sls_price stored as INT - should be DECIMAL in Silver layer")

print("✅ crm_sales tests completed")

## 4. bike_store.bronze.erp_category
Category data from ERP system

In [0]:
table_name = "bike_store.bronze.erp_category"
df = spark.table(table_name)

print(f"Testing {table_name}...")
print(f"Total records: {df.count():,}\n")

# 1. Completeness checks
required_cols = ["ID", "CAT"]
check_completeness(df, table_name, required_cols)

# 2. Schema consistency
expected_schema = {
    "ID": "string",
    "CAT": "string",
    "SUBCAT": "string",
    "MAINTENANCE": "string",
    "file_name": "string",
    "ingest_datetime": "timestamp"
}
check_schema(df, table_name, expected_schema)

# 3. Data quality checks
# Check for duplicate category IDs
dup_count = df.groupBy("ID").count().filter("count > 1").count()
if dup_count > 0:
    log_test(table_name, "Uniqueness", "ID unique", "FAIL", f"{dup_count} duplicate IDs")
else:
    log_test(table_name, "Uniqueness", "ID unique", "PASS", "")

# Check naming convention consistency
log_test(table_name, "Naming Convention", "Column names", "INFO", 
        "ERP tables use UPPERCASE - inconsistent with CRM (lowercase with underscores)")

print("✅ erp_category tests completed")

## 5. bike_store.bronze.erp_customer
Customer data from ERP system

In [0]:
table_name = "bike_store.bronze.erp_customer"
df = spark.table(table_name)

print(f"Testing {table_name}...")
print(f"Total records: {df.count():,}\n")

# 1. Completeness checks
required_cols = ["CID"]
check_completeness(df, table_name, required_cols)

# 2. Schema consistency
expected_schema = {
    "CID": "string",
    "BDATE": "date",
    "GEN": "string",
    "file_name": "string",
    "ingest_datetime": "timestamp"
}
check_schema(df, table_name, expected_schema)

# 3. Range validations
# Birth date should be in the past and reasonable
from datetime import date
check_range(df, table_name, "BDATE", 
           min_val=date(1900, 1, 1), 
           max_val=date.today())

# 4. Data quality checks
# Check for duplicate customer IDs
dup_count = df.groupBy("CID").count().filter("count > 1").count()
if dup_count > 0:
    log_test(table_name, "Uniqueness", "CID unique", "FAIL", f"{dup_count} duplicate IDs")
else:
    log_test(table_name, "Uniqueness", "CID unique", "PASS", "")

# Check gender values
valid_genders = df.filter(F.col("GEN").isin(["M", "F", "Male", "Female"])).count()
total = df.filter(F.col("GEN").isNotNull()).count()
if valid_genders < total:
    log_test(table_name, "Validity", "GEN valid values", "WARN", 
            f"{total - valid_genders} invalid gender values")
else:
    log_test(table_name, "Validity", "GEN valid values", "PASS", "")

print("✅ erp_customer tests completed")

## 6. bike_store.bronze.erp_location
Location data from ERP system

In [0]:
table_name = "bike_store.bronze.erp_location"
df = spark.table(table_name)

print(f"Testing {table_name}...")
print(f"Total records: {df.count():,}\n")

# 1. Completeness checks
required_cols = ["CID", "CNTRY"]
check_completeness(df, table_name, required_cols)

# 2. Schema consistency
expected_schema = {
    "CID": "string",
    "CNTRY": "string",
    "file_name": "string",
    "ingest_datetime": "timestamp"
}
check_schema(df, table_name, expected_schema)

# 3. Relationship validations
# Check FK to erp_customer
erp_customer_df = spark.table("bike_store.bronze.erp_customer")
valid_customer_ids = erp_customer_df.select("CID").distinct()
orphaned_locations = df.join(valid_customer_ids, df.CID == valid_customer_ids.CID, "left_anti").count()
if orphaned_locations > 0:
    log_test(table_name, "Referential Integrity", "CID FK valid", "FAIL", 
            f"{orphaned_locations} locations with invalid customer IDs")
else:
    log_test(table_name, "Referential Integrity", "CID FK valid", "PASS", "")

# 4. Data quality checks
# Check country code format (should be 2-3 letters)
invalid_country = df.filter(
    (F.col("CNTRY").isNotNull()) & 
    ((F.length(F.col("CNTRY")) < 2) | (F.length(F.col("CNTRY")) > 3))
).count()
if invalid_country > 0:
    log_test(table_name, "Validity", "CNTRY format", "WARN", 
            f"{invalid_country} records with invalid country code length")
else:
    log_test(table_name, "Validity", "CNTRY format", "PASS", "")

print("✅ erp_location tests completed")

## Test Summary Report
Comprehensive overview of all data quality test results

In [0]:
# Convert test results to DataFrame
results_df = spark.createDataFrame(pd.DataFrame(test_results))

# Display full results
print("="*100)
print("DETAILED TEST RESULTS")
print("="*100)
display(results_df.orderBy("table", "category", "test"))

# Summary by status
print("\n" + "="*100)
print("SUMMARY BY STATUS")
print("="*100)
summary_status = results_df.groupBy("status").count().orderBy("status")
display(summary_status)

# Summary by table
print("\n" + "="*100)
print("SUMMARY BY TABLE")
print("="*100)
summary_table = results_df.groupBy("table", "status").count().orderBy("table", "status")
display(summary_table)

# Summary by category
print("\n" + "="*100)
print("SUMMARY BY CATEGORY")
print("="*100)
summary_category = results_df.groupBy("category", "status").count().orderBy("category", "status")
display(summary_category)

# Failed tests
failed_tests = results_df.filter(F.col("status") == "FAIL")
failed_count = failed_tests.count()

print("\n" + "="*100)
if failed_count > 0:
    print(f"⚠️  CRITICAL: {failed_count} TESTS FAILED")
    print("="*100)
    display(failed_tests.select("table", "category", "test", "details"))
else:
    print("✅ ALL TESTS PASSED (no failures detected)")
    print("="*100)

# Warnings
warn_tests = results_df.filter(F.col("status") == "WARN")
warn_count = warn_tests.count()

if warn_count > 0:
    print("\n" + "="*100)
    print(f"⚠️  {warn_count} WARNINGS DETECTED")
    print("="*100)
    display(warn_tests.select("table", "category", "test", "details"))

print("\n" + "="*100)
print("TEST EXECUTION COMPLETED")
print("="*100)
print(f"Total tests executed: {len(test_results)}")
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## Export Critical Failures to CSV
Export failed tests to CSV file for reporting and tracking

In [0]:
# Filter for failed tests only
failed_tests_df = results_df.filter(F.col("status") == "FAIL")

# Select relevant columns and convert to Pandas for CSV export
failed_export = failed_tests_df.select(
    "table", 
    "category", 
    "test", 
    "details",
    "timestamp"
).toPandas()

# Define output path
output_path = "/Workspace/Users/adriano.study31@gmail.com/Project_bikestore/critical_failures_bronze.csv"

# Save to CSV
failed_export.to_csv(output_path, index=False)

print(f"✅ Critical failures exported successfully!")
print(f"📁 File location: {output_path}")
print(f"📊 Total failures exported: {len(failed_export)}")
print(f"\nPreview of exported data:")
display(failed_export)